# 🍎 Real-Time Fruit Detection - Training & Evaluation

**Model**: YOLOv8s (fine-tuned)  
**Dataset**: 6 kelas - Apple, Banana, Grape, Orange, Pineapple, Watermelon  
**Total Data**: 7,108 train | 914 val | 457 test

---

## 📋 Langkah-langkah:
1. Install & Import Dependencies
2. Explorasi Dataset
3. Training Model YOLOv8s
4. Evaluasi & Visualisasi Hasil
5. Prediksi pada Gambar Test
6. Export Model (ONNX)

## 1️⃣ Install & Import Dependencies

In [ ]:
# Install library yang dibutuhkan (jalankan sekali saja)
!pip install ultralytics opencv-python matplotlib seaborn PyYAML Pillow --quiet

In [ ]:
import os
import yaml
import random
from pathlib import Path
from collections import Counter

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from PIL import Image
from ultralytics import YOLO

# Style plot
plt.style.use('dark_background')
sns.set_palette('bright')

# Project paths
PROJECT_DIR = Path('.').resolve()
DATA_YAML = PROJECT_DIR / 'data.yaml'
PRETRAINED_MODEL = PROJECT_DIR / 'yolov8s.pt'

print(f'📁 Project Directory : {PROJECT_DIR}')
print(f'📄 Data YAML         : {DATA_YAML}')
print(f'🧠 Pretrained Model  : {PRETRAINED_MODEL}')
print(f'\n✅ Semua library berhasil di-import!')

## 2️⃣ Explorasi Dataset

In [ ]:
# Load data.yaml
with open(DATA_YAML, 'r') as f:
    data_config = yaml.safe_load(f)

class_names = data_config['names']
num_classes = data_config['nc']

print(f'📊 Dataset Configuration:')
print(f'   Jumlah Kelas : {num_classes}')
print(f'   Nama Kelas   : {class_names}')
print(f'   Train Path   : {data_config["train"]}')
print(f'   Val Path     : {data_config["val"]}')
print(f'   Test Path    : {data_config["test"]}')

In [ ]:
# Hitung jumlah gambar per split
splits = {
    'Train': PROJECT_DIR / 'train' / 'images',
    'Validation': PROJECT_DIR / 'valid' / 'images',
    'Test': PROJECT_DIR / 'test' / 'images',
}

split_counts = {}
for split_name, split_path in splits.items():
    count = len(list(split_path.glob('*')))
    split_counts[split_name] = count
    print(f'  {split_name:12s}: {count:,} gambar')

# Visualisasi distribusi data
fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
bars = ax.bar(split_counts.keys(), split_counts.values(), color=colors,
              edgecolor='white', linewidth=0.5, width=0.6)

for bar, count in zip(bars, split_counts.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'{count:,}', ha='center', va='bottom', fontsize=14, fontweight='bold', color='white')

ax.set_title('📊 Distribusi Data per Split', fontsize=16, pad=15)
ax.set_ylabel('Jumlah Gambar', fontsize=12)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# Analisis distribusi kelas dari label files
label_dir = PROJECT_DIR / 'train' / 'labels'
class_counter = Counter()

for label_file in label_dir.glob('*.txt'):
    with open(label_file, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                class_id = int(parts[0])
                class_counter[class_id] += 1

# Sort by class_id
sorted_classes = sorted(class_counter.items())
labels = [class_names[cid] for cid, _ in sorted_classes]
counts = [cnt for _, cnt in sorted_classes]

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
emoji_colors = ['#FF4444', '#FFD700', '#9B59B6', '#FF8C00', '#2ECC71', '#27AE60']
bars = ax.barh(labels, counts, color=emoji_colors[:len(labels)],
               edgecolor='white', linewidth=0.5, height=0.6)

for bar, count in zip(bars, counts):
    ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height()/2,
            f'{count:,}', va='center', fontsize=12, fontweight='bold', color='white')

ax.set_title('🏷️ Distribusi Kelas pada Training Set', fontsize=16, pad=15)
ax.set_xlabel('Jumlah Objek', fontsize=12)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print('\n📊 Detail per kelas:')
total = sum(counts)
for label, count in zip(labels, counts):
    pct = count / total * 100
    print(f'   {label:12s}: {count:>6,} objek ({pct:.1f}%)')
print(f'   {"TOTAL":12s}: {total:>6,} objek')

In [ ]:
# Tampilkan sample gambar dari dataset
train_images_dir = PROJECT_DIR / 'train' / 'images'
sample_images = random.sample(list(train_images_dir.glob('*')), min(12, len(list(train_images_dir.glob('*')))))

fig, axes = plt.subplots(2, 6, figsize=(20, 7))
fig.suptitle('🖼️ Sample Gambar dari Training Set', fontsize=18, y=1.02)

for idx, (ax, img_path) in enumerate(zip(axes.flat, sample_images)):
    img = Image.open(img_path)
    ax.imshow(img)
    ax.set_title(img_path.name[:20], fontsize=8, color='white')
    ax.axis('off')

# Hide empty subplots
for idx in range(len(sample_images), len(axes.flat)):
    axes.flat[idx].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Tampilkan sample gambar DENGAN bounding box
def show_image_with_boxes(img_path, label_path, class_names, ax):
    """Tampilkan gambar dengan bounding box dari label YOLO."""
    img = Image.open(img_path)
    w, h = img.size
    ax.imshow(img)
    
    colors_map = ['#FF4444', '#FFD700', '#9B59B6', '#FF8C00', '#2ECC71', '#27AE60']
    
    if label_path.exists():
        with open(label_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    cls_id = int(parts[0])
                    cx, cy, bw, bh = map(float, parts[1:5])
                    
                    # Convert YOLO format to pixel coordinates
                    x1 = (cx - bw/2) * w
                    y1 = (cy - bh/2) * h
                    box_w = bw * w
                    box_h = bh * h
                    
                    color = colors_map[cls_id % len(colors_map)]
                    rect = patches.Rectangle((x1, y1), box_w, box_h,
                                            linewidth=2, edgecolor=color, facecolor='none')
                    ax.add_patch(rect)
                    ax.text(x1, y1 - 5, class_names[cls_id],
                           fontsize=8, color=color, fontweight='bold',
                           bbox=dict(boxstyle='round,pad=0.2', facecolor='black', alpha=0.7))
    ax.axis('off')


# Pilih sample acak
sample_imgs = random.sample(list(train_images_dir.glob('*')), min(6, len(list(train_images_dir.glob('*')))))

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('🎯 Sample Gambar dengan Bounding Box (Ground Truth)', fontsize=18, y=1.02)

for ax, img_path in zip(axes.flat, sample_imgs):
    label_path = PROJECT_DIR / 'train' / 'labels' / (img_path.stem + '.txt')
    show_image_with_boxes(img_path, label_path, class_names, ax)

for idx in range(len(sample_imgs), len(axes.flat)):
    axes.flat[idx].axis('off')

plt.tight_layout()
plt.show()

## 3️⃣ Training Model YOLOv8s

Fine-tune model YOLOv8s dengan dataset buah kita.

> ⚠️ **Tips**: Jika tidak punya GPU, ganti `device='cpu'` (tapi akan sangat lambat).
> Disarankan menggunakan GPU (NVIDIA) atau Google Colab.

In [ ]:
# Cek ketersediaan GPU
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_mem / 1024**3
    print(f'✅ GPU tersedia: {gpu_name} ({gpu_memory:.1f} GB)')
    DEVICE = '0'
else:
    print('⚠️ GPU tidak tersedia, menggunakan CPU (training akan lambat!)')
    DEVICE = 'cpu'

print(f'🔧 Device yang digunakan: {DEVICE}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  HYPERPARAMETERS - Sesuaikan jika perlu                     ║
# ╚══════════════════════════════════════════════════════════════╝

EPOCHS = 50          # Jumlah epoch (50-100 sudah cukup untuk dataset ini)
BATCH_SIZE = 16      # Kurangi jika GPU kehabisan memory (misal: 8)
IMG_SIZE = 640       # Ukuran input gambar
PATIENCE = 15        # Early stopping patience
WORKERS = 4          # Data loader workers

print(f'📋 Hyperparameters:')
print(f'   Epochs     : {EPOCHS}')
print(f'   Batch Size : {BATCH_SIZE}')
print(f'   Image Size : {IMG_SIZE}')
print(f'   Patience   : {PATIENCE}')
print(f'   Workers    : {WORKERS}')
print(f'   Device     : {DEVICE}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  TRAINING                                                   ║
# ╚══════════════════════════════════════════════════════════════╝

# Load model pretrained
model = YOLO(str(PRETRAINED_MODEL))

# Mulai training
results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMG_SIZE,
    device=DEVICE,
    workers=WORKERS,
    project=str(PROJECT_DIR / 'fruit_detection_runs'),
    name='train',
    exist_ok=True,
    # Augmentasi
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    # Optimizer
    optimizer='auto',
    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    # Early stopping
    patience=PATIENCE,
    # Logging
    verbose=True,
    plots=True,
)

print('\n' + '=' * 60)
print('  ✅ TRAINING SELESAI!')
print('=' * 60)

## 4️⃣ Evaluasi & Visualisasi Hasil Training

In [ ]:
# Load model terbaik dari hasil training
BEST_MODEL_PATH = PROJECT_DIR / 'fruit_detection_runs' / 'train' / 'weights' / 'best.pt'

if BEST_MODEL_PATH.exists():
    best_model = YOLO(str(BEST_MODEL_PATH))
    print(f'✅ Best model loaded: {BEST_MODEL_PATH}')
else:
    print(f'❌ Model tidak ditemukan: {BEST_MODEL_PATH}')
    print('   Jalankan cell training di atas terlebih dahulu!')

In [ ]:
# Tampilkan grafik training (dari folder results)
results_dir = PROJECT_DIR / 'fruit_detection_runs' / 'train'

# Cari dan tampilkan plot yang dihasilkan YOLOv8
plot_files = [
    'results.png',
    'confusion_matrix.png',
    'confusion_matrix_normalized.png',
    'F1_curve.png',
    'P_curve.png',
    'R_curve.png',
    'PR_curve.png',
]

for plot_file in plot_files:
    plot_path = results_dir / plot_file
    if plot_path.exists():
        img = Image.open(plot_path)
        fig, ax = plt.subplots(figsize=(14, 8))
        ax.imshow(img)
        ax.set_title(plot_file.replace('.png', '').replace('_', ' ').title(),
                    fontsize=16, pad=10)
        ax.axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print(f'⚠️ Plot tidak ditemukan: {plot_file}')

In [ ]:
# Evaluasi pada Validation Set
print('📊 Evaluasi pada Validation Set...')
val_results = best_model.val(
    data=str(DATA_YAML),
    split='val',
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    plots=True,
    project=str(PROJECT_DIR / 'fruit_detection_runs'),
    name='val_eval',
    exist_ok=True,
)

print(f'\n📈 Validation Results:')
print(f'   mAP50      : {val_results.box.map50:.4f}')
print(f'   mAP50-95   : {val_results.box.map:.4f}')
print(f'   Precision  : {val_results.box.mp:.4f}')
print(f'   Recall     : {val_results.box.mr:.4f}')

In [ ]:
# Evaluasi pada Test Set
print('📊 Evaluasi pada Test Set...')
test_results = best_model.val(
    data=str(DATA_YAML),
    split='test',
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    plots=True,
    project=str(PROJECT_DIR / 'fruit_detection_runs'),
    name='test_eval',
    exist_ok=True,
)

print(f'\n📈 Test Results:')
print(f'   mAP50      : {test_results.box.map50:.4f}')
print(f'   mAP50-95   : {test_results.box.map:.4f}')
print(f'   Precision  : {test_results.box.mp:.4f}')
print(f'   Recall     : {test_results.box.mr:.4f}')

In [ ]:
# Ringkasan hasil evaluasi
print('\n' + '=' * 65)
print('  📊 RINGKASAN EVALUASI MODEL')
print('=' * 65)
print(f'  {"Metric":<20s} {"Validation":>12s} {"Test":>12s}')
print(f'  {"-"*20} {"-"*12} {"-"*12}')
print(f'  {"mAP50":<20s} {val_results.box.map50:>12.4f} {test_results.box.map50:>12.4f}')
print(f'  {"mAP50-95":<20s} {val_results.box.map:>12.4f} {test_results.box.map:>12.4f}')
print(f'  {"Precision":<20s} {val_results.box.mp:>12.4f} {test_results.box.mp:>12.4f}')
print(f'  {"Recall":<20s} {val_results.box.mr:>12.4f} {test_results.box.mr:>12.4f}')
print('=' * 65)

## 5️⃣ Prediksi pada Gambar Test

In [ ]:
# Prediksi pada beberapa gambar test
test_images_dir = PROJECT_DIR / 'test' / 'images'
test_images = list(test_images_dir.glob('*'))
sample_test = random.sample(test_images, min(8, len(test_images)))

fig, axes = plt.subplots(2, 4, figsize=(24, 12))
fig.suptitle('🔍 Prediksi Model pada Gambar Test', fontsize=20, y=1.02)

for ax, img_path in zip(axes.flat, sample_test):
    # Jalankan prediksi
    results = best_model.predict(
        source=str(img_path),
        imgsz=IMG_SIZE,
        conf=0.5,
        device=DEVICE,
        verbose=False,
    )
    
    # Ambil gambar hasil (sudah ada bounding box)
    result_img = results[0].plot()
    result_img_rgb = cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB)
    
    ax.imshow(result_img_rgb)
    
    # Jumlah deteksi
    n_detections = len(results[0].boxes) if results[0].boxes is not None else 0
    ax.set_title(f'{img_path.name[:25]}\n({n_detections} deteksi)', fontsize=10)
    ax.axis('off')

for idx in range(len(sample_test), len(axes.flat)):
    axes.flat[idx].axis('off')

plt.tight_layout()
plt.show()

## 6️⃣ Export Model (Opsional)

Export model ke format ONNX untuk deployment di production.

In [ ]:
# Export ke ONNX (opsional)
# Uncomment baris di bawah untuk export

# best_model.export(format='onnx', imgsz=640, dynamic=True)
# print('✅ Model berhasil di-export ke format ONNX!')

print('💡 Untuk export, uncomment kode di atas dan jalankan cell ini.')
print('   Format yang didukung: onnx, torchscript, tflite, coreml, dll.')

---

## ✅ Selesai!

### Langkah Selanjutnya:
1. **Real-time Detection**: Jalankan `realtime_fruit_detection.py` untuk deteksi lewat webcam
   ```bash
   python realtime_fruit_detection.py
   ```

2. **Improve Model**: Jika akurasi belum memuaskan:
   - Tambah epoch (100-200)
   - Gunakan model lebih besar (yolov8m.pt atau yolov8l.pt)
   - Tambah data training
   - Tune hyperparameters

3. **Deploy**: Export model ke ONNX untuk deployment di edge device